In [1]:
# Get raw data
with open('input/20.txt', 'r') as f:
    rawinput = f.read().strip()

In [2]:
# Part 1
get_edges = lambda x: [(z:=''.join(x))[:10], z[9::10], z[99:89:-1], z[90::-10]]

tiles = {int(i[5:9]): j
         for i in rawinput.split('\n\n')
         for j in [i[11:].split('\n')]}
tile_edges = {k: {s:get_edges(v[::s]) for s in [1,-1]}
              for k,v in tiles.items()}

edge_match = {i: {j: [[(k,l,f) 
                       for k in tile_edges if k!=i
                       for l in [1,-1] for f in [0,1,2,3] 
                       if tile_edges[k][l][f]==tile_edges[i][j][e][::-1]]
                      for e in [0,1,2,3]]
                  for j in [1,-1]}
              for i in tile_edges}
corners = [i for i in edge_match
           if sum([len(j)==0 for j in edge_match[i][1]])==2]

[(z:=z*j if i else j) for i,j in enumerate(corners)][-1]

21599955909991

In [3]:
# Part 2
class Cells(object):
    def __init__(self, rows, cols, topleft):
        self.rows = rows+2
        self.cols = cols+2
        self.tile = [[-1 if {i,j}&{0,13} else None
                      for j in range(self.rows)] 
                     for i in range(self.cols)]
        self.tile[1][1] = topleft
        self.side = [[-1 if {i,j}&{0,13} else None
                      for j in range(self.rows)] 
                     for i in range(self.cols)]
        self.orient = [[-1 if {i,j}&{0,13} else None
                        for j in range(self.rows)] 
                       for i in range(self.cols)]
        
    def solve(self):
        for r in range(1, self.rows-1):
            for c in range(1, self.rows-1):
                self.orient_tile(r,c)
        
    def orient_tile(self, row, col):
        tile = self.tile[row][col]
        tomatch = [self.tile[row+j][col+k]
                   for i,(j,k) in enumerate([[-1,0],[0,1],[1,0],[0,-1]])]
        side,orient = [[j,k]
                       for j in [1,-1]
                       for k in [0,1,2,3]
                       if all([n is None or m==n
                               for m,n in zip([z[0][0] if z else -1
                                               for z in [edge_match[tile][j][(l-k)%4] 
                                                         for l in [0,1,2,3]]],
                                              tomatch)])][0]
        self.add_tile(row, col, tile, side, orient)        

    def add_tile(self, row, col, tile, side, orient):
        self.tile[row][col] = tile
        self.side[row][col] = side
        self.orient[row][col] = orient
        for i,(j,k) in enumerate([[-1,0],[0,1],[1,0],[0,-1]]):
            if self.tile[row+j][col+k] is None:
                self.tile[row+j][col+k] = (z[0][0] 
                                           if (z:=edge_match[tile][side][(i-orient)%4]) 
                                           else -1)
                
cells = Cells(12,12,corners[0])
cells.solve()

rotate = lambda x,y=0: rotate([''.join(i[::-1]) for i in zip(*x)],y-1) if y else x

image = [((rr-1)*8+i,(cc-1)*8+j)
         for rr in range(1, cells.rows-1)
         for cc in range(1, cells.cols-1)
         for i,r in enumerate(rotate(tiles[cells.tile[rr][cc]]
                                     [::cells.side[rr][cc]], 
                                     cells.orient[rr][cc])[1:-1])
         for j,c in enumerate(r[1:-1])
         if c=='#']

monster_raw = """                  # 
#    ##    ##    ###
 #  #  #  #  #  #   """
monsters = [sorted([*zip(*[[l-min(k) for l in k] for k in zip(*z)])])
            for g in [1,-1]
            for h in [0,1,2,3]
            for z in [[(g*i,j)
                       for i,r in enumerate(rotate(monster_raw.split('\n'),h))
                       for j,c in enumerate(r)
                       if c=='#']]]

len(image) - len({p
                  for i in range(96)
                  for j in range(96) 
                  for m in range(len(monsters))
                  if (z:={(i+k,j+l) for k,l in monsters[m]}).issubset(image)
                  for p in z})

2495